# RAG Based LLM Implementation (Baseline)

## Problem Statement
Readers may want to understand more about the characters, events, and motivations in a book, but they may not always have someone available to discuss it with. Finding passages that answer their questions can also require searching through different parts of the book.

This project builds a RAG-based book assistant that allows readers to ask questions about a collection of books. It retrieves relevant passages and generates answers with source references so readers can check the supporting evidence.

## Dataset

The dataset consists of four English-language books from Project Gutenberg, stored as UTF-8 .txt files, with a total file size of approximately 2 MB. Preprocessing includes removing lines containing only asterisks, removing empty [Illustration] markers, normalizing line endings, reducing excessive blank lines, and retaining only the text between the Project Gutenberg START and END markers.

## Tools and Models

The system provides a text-based menu within a Jupyter notebook for loading documents and asking questions. It runs locally without a graphical application or cloud deployment.
| Component | Model or tool | Reason for selection |
|---|---|---|
| LLM | `Qwen/Qwen2.5-1.5B-Instruct` | Selected as a relatively small instruction-tuned model for local execution without a paid API. |
| Embedding model | `BAAI/bge-base-en-v1.5` | Selected to generate embeddings for semantic retrieval over the English-language book collection. |
| Vector store | ChromaDB | Selected for local persistence and integration with LangChain. |
| Framework | LangChain | Used to connect document loading, chunking, retrieval, and answer generation. |

The models are downloaded from Hugging Face and executed locally. Internet access is needed for the initial download. Model selection prioritizes local accessibility and implementation simplicity; these models have not been established as the best-performing options through comparative testing.
Saved examples include the original questions, retrieved passages, source references, generated answers, and manual reviews so that reviewers can inspect the system’s behavior without running the models.

### Configurable

In [ ]:
DOCS_PATH = "./docs/"
DB_PATH = "./db/ChromaDB/"
COLLECTION_NAME = "books_dedup"

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
DEVICE = ""

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 0
TOP_K = 5
MAX_NEW_TOKENS = 512

## Evaluation Result

## Retrieval Result

I chose Hit@5 and MRR@5 because the system retrieves five passages for each question. Hit@5 checks whether a reference passage appears in those five results. MRR@5 checks how high the first reference passage appears in the results. These metrics are suitable for an initial evaluation using factual questions and selected reference chunks. However, they do not check whether every retrieved passage is relevant or whether all the evidence needed for a question has been retrieved.

Question: Why does Alice follow the White Rabbit?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 41 
3. alice_in_wonderland.txt — Chunk 2 (Referenced)
4. alice_in_wonderland.txt — Chunk 158 
5. alice_in_wonderland.txt — Chunk 141 

Hit: 1

Reciprocal rank: 0.333

Question: What opens the small door behind the curtain?
1. alice_in_wonderland.txt — Chunk 8 (Referenced)
2. sherlock_holmes.txt — Chunk 723 
3. sherlock_holmes.txt — Chunk 52 
4. sherlock_holmes.txt — Chunk 645 
5. alice_in_wonderland.txt — Chunk 9 

Hit: 1

Reciprocal rank: 1.000

Question: Why can't Alice climb the table leg to reach the key?
1. alice_in_wonderland.txt — Chunk 13 (Referenced)
2. alice_in_wonderland.txt — Chunk 8 
3. alice_in_wonderland.txt — Chunk 48 
4. alice_in_wonderland.txt — Chunk 4 
5. alice_in_wonderland.txt — Chunk 3 

Hit: 1

Reciprocal rank: 1.000

Question: Who wakes Alice at the end of her adventure?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 51 
3. alice_in_wonderland.txt — Chunk 40 
4. alice_in_wonderland.txt — Chunk 96 
5. alice_in_wonderland.txt — Chunk 52 

Hit: 0

Reciprocal rank: 0.000

Question: Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?
1. sherlock_holmes.txt — Chunk 425 (Referenced)
2. sherlock_holmes.txt — Chunk 451 
3. sherlock_holmes.txt — Chunk 587 
4. sherlock_holmes.txt — Chunk 406 
5. sherlock_holmes.txt — Chunk 368 

Hit: 1
Reciprocal rank: 1.000

Retrieval evaluation

Hit Rate: 80.00%

MRR: 0.667

## Generation Result

I chose faithfulness to check whether the model’s answers are supported by the retrieved documents. I chose answer correctness to check whether the answers match the reference answers in meaning and facts. Together, these metrics help evaluate whether the system gives correct answers based on the documents. I also review the outputs because the evaluator model can make mistakes.

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>user_input</th>
      <th>response</th>
      <th>reference</th>
      <th>faithfulness</th>
      <th>answer_correctness</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Why does Alice follow the White Rabbit?</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>1.000000</td>
      <td>0.717572</td>
    </tr>
    <tr>
      <th>1</th>
      <td>What opens the small door behind the curtain?</td>
      <td>Alice opens the small door behind the curtain.</td>
      <td>A little golden key.</td>
      <td>1.000000</td>
      <td>0.899673</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Why can't Alice climb the table leg to reach t...</td>
      <td>Alice cannot climb the table leg to reach the ...</td>
      <td>Because the table leg is too slippery.</td>
      <td>0.000000</td>
      <td>0.656221</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Who wakes Alice at the end of her adventure?</td>
      <td>Alice doesn't wake anyone at the end of her ad...</td>
      <td>Her sister.</td>
      <td>0.333333</td>
      <td>0.090247</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Who does Sherlock Holmes introduce as his frie...</td>
      <td>Sherlock Holmes introduces Dr. Watson as his f...</td>
      <td>Dr. Watson.</td>
      <td>1.000000</td>
      <td>0.914123</td>
    </tr>
  </tbody>
</table>
</div>

Average scores:

faithfulness          0.666667

answer_correctness    0.655567

## Result

### Question: Who does Alice follow down the rabbit hole?

Found 5 documents
Answer: Alice follows the rabbit hole down.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Either the well was very deep, or she fell very slowly, for she had
plenty of time as she went down to look about her and to wonder what
was going to happen next. First, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-shelves; here and there she saw maps and pictures
hung upon pegs. She took down a jar from one of the shelves as she
passed; it was labelled “ORANGE MARMALADE”, but to her great
disappointment it was empty: she did not like to drop the jar for fear
of killing somebody underneath, so managed to put it into one of the
cupboards as she fell past it.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 0
Evidence: Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence

CHAPTER I.
Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 43
Evidence: “How queer it seems,” Alice said to herself, “to be going messages for
a rabbit! I suppose Dinah’ll be sending me on messages next!” And she
began fancying the sort of thing that would happen: “‘Miss Alice! Come
here directly, and get ready for your walk!’ ‘Coming in a minute,
nurse! But I’ve got to see that the mouse doesn’t get out.’ Only I
don’t think,” Alice went on, “that they’d let Dinah stop in the house
if it began ordering people about like that!”

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 158
Evidence: Alice watched the White Rabbit as he fumbled over the list, feeling
very curious to see what the next witness would be like, “—for they
haven’t got much evidence _yet_,” she said to herself. Imagine her
surprise, when the White Rabbit read out, at the top of his shrill
little voice, the name “Alice!”

CHAPTER XII.
Alice’s Evidence

“Here!” cried Alice, quite forgetting in the flurry of the moment how
large she had grown in the last few minutes, and she jumped up in such
a hurry that she tipped over the jury-box with the edge of her skirt,
upsetting all the jurymen on to the heads of the crowd below, and there
they lay sprawling about, reminding her very much of a globe of
goldfish she had accidentally upset the week before.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

### Review
The answer is correct according to the book. Chunk 2 directly describes Alice following the Rabbit into the rabbit hole, although it does not explicitly identify him as the White Rabbit within that chunk. Chunk 3 provides context about her fall, while the remaining passages contribute little to answering the question. Retrieval finds the relevant event, but the character’s introduction would provide clearer support for his identity.

### Question: Why does Alice follow the White Rabbit?

Answer: Alice follows the White Rabbit because she is curious about him and wants to find out more about his whereabouts.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door, she walked sadly down the middle, wondering how she was ever to
get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 41
Evidence: It was the White Rabbit, trotting slowly back again, and looking
anxiously about as it went, as if it had lost something; and she heard
it muttering to itself “The Duchess! The Duchess! Oh my dear paws! Oh
my fur and whiskers! She’ll get me executed, as sure as ferrets are
ferrets! Where _can_ I have dropped them, I wonder?” Alice guessed in a
moment that it was looking for the fan and the pair of white kid
gloves, and she very good-naturedly began hunting about for them, but
they were nowhere to be seen—everything seemed to have changed since
her swim in the pool, and the great hall, with the glass table and the
little door, had vanished completely.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 158
Evidence: Alice watched the White Rabbit as he fumbled over the list, feeling
very curious to see what the next witness would be like, “—for they
haven’t got much evidence _yet_,” she said to herself. Imagine her
surprise, when the White Rabbit read out, at the top of his shrill
little voice, the name “Alice!”

CHAPTER XII.
Alice’s Evidence

“Here!” cried Alice, quite forgetting in the flurry of the moment how
large she had grown in the last few minutes, and she jumped up in such
a hurry that she tipped over the jury-box with the edge of her skirt,
upsetting all the jurymen on to the heads of the crowd below, and there
they lay sprawling about, reminding her very much of a globe of
goldfish she had accidentally upset the week before.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 141
Evidence: “No, no! The adventures first,” said the Gryphon in an impatient tone:
“explanations take such a dreadful time.”

So Alice began telling them her adventures from the time when she first
saw the White Rabbit. She was a little nervous about it just at first,
the two creatures got so close to her, one on each side, and opened
their eyes and mouths so _very_ wide, but she gained courage as she
went on. Her listeners were perfectly quiet till she got to the part
about her repeating “_You are old, Father William_,” to the
Caterpillar, and the words all coming different, and then the Mock
Turtle drew a long breath, and said “That’s very curious.”

“It’s all about as curious as it can be,” said the Gryphon.

“It all came different!” the Mock Turtle repeated thoughtfully. “I
should like to hear her try and repeat something now. Tell her to
begin.” He looked at the Gryphon as if he thought it had some kind of
authority over Alice.

### Review
The answer correctly identifies curiosity as Alice’s motivation, which is explicitly supported by chunk 2. However, it omits the specific reason for her curiosity which is seeing the Rabbit take a watch out of its waistcoat pocket. The additional claim that she wants to find out more about his whereabouts is not clearly established by the retrieved evidence. Retrieval includes the necessary passage, but the answer could be more precise and better grounded.

### Question: Does Alice eventually get out of Wonderland?

Answer: No, Alice does not eventually get out of Wonderland.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 86
Evidence: “I should like it very much,” said Alice, “but I haven’t been invited
yet.”

“You’ll see me there,” said the Cat, and vanished.

Alice was not much surprised at this, she was getting so used to queer
things happening. While she was looking at the place where it had been,
it suddenly appeared again.

“By-the-bye, what became of the baby?” said the Cat. “I’d nearly
forgotten to ask.”

“It turned into a pig,” Alice quietly said, just as if it had come back
in a natural way.

“I thought it would,” said the Cat, and vanished again.

Alice waited a little, half expecting to see it again, but it did not
appear, and after a minute or two she walked on in the direction in
which the March Hare was said to live. “I’ve seen hatters before,” she
said to herself; “the March Hare will be much the most interesting, and
perhaps as this is May it won’t be raving mad—at least not so mad as it
was in March.” As she said this, she looked up, and there was the Cat
again, sitting on a branch of a tree.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door, she walked sadly down the middle, wondering how she was ever to
get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 117
Evidence: By the time she had caught the flamingo and brought it back, the fight
was over, and both the hedgehogs were out of sight: “but it doesn’t
matter much,” thought Alice, “as all the arches are gone from this side
of the ground.” So she tucked it away under her arm, that it might not
escape again, and went back for a little more conversation with her
friend.

When she got back to the Cheshire Cat, she was surprised to find quite
a large crowd collected round it: there was a dispute going on between
the executioner, the King, and the Queen, who were all talking at once,
while all the rest were quite silent, and looked very uncomfortable.

The moment Alice appeared, she was appealed to by all three to settle
the question, and they repeated their arguments to her, though, as they
all spoke at once, she found it very hard indeed to make out exactly
what they said.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 40
Evidence: “I wish I hadn’t mentioned Dinah!” she said to herself in a melancholy
tone. “Nobody seems to like her, down here, and I’m sure she’s the best
cat in the world! Oh, my dear Dinah! I wonder if I shall ever see you
any more!” And here poor Alice began to cry again, for she felt very
lonely and low-spirited. In a little while, however, she again heard a
little pattering of footsteps in the distance, and she looked up
eagerly, half hoping that the Mouse had changed his mind, and was
coming back to finish his story.

CHAPTER IV.
The Rabbit Sends in a Little Bill

### Review
The answer is incorrect according to the book, which ends with Alice waking up beside her sister. The retrieved passages describe earlier events and do not include the ending. Although chunks 7 and 40 show Alice’s uncertainty about leaving, they do not establish that she never leaves. Retrieval misses the necessary evidence, and generation turns insufficient information into an unsupported negative answer instead of acknowledging uncertainty.

### Question: How does Alice get out of Wonderland?

Answer: Alice gets out of Wonderland by climbing back through the rabbit hole.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door, she walked sadly down the middle, wondering how she was ever to
get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 0
Evidence: Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence

CHAPTER I.
Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Either the well was very deep, or she fell very slowly, for she had
plenty of time as she went down to look about her and to wonder what
was going to happen next. First, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-shelves; here and there she saw maps and pictures
hung upon pegs. She took down a jar from one of the shelves as she
passed; it was labelled “ORANGE MARMALADE”, but to her great
disappointment it was empty: she did not like to drop the jar for fear
of killing somebody underneath, so managed to put it into one of the
cupboards as she fell past it.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 51
Evidence: “Oh! So Bill’s got to come down the chimney, has he?” said Alice to
herself. “Shy, they seem to put everything upon Bill! I wouldn’t be in
Bill’s place for a good deal: this fireplace is narrow, to be sure; but
I _think_ I can kick a little!”

She drew her foot as far down the chimney as she could, and waited till
she heard a little animal (she couldn’t guess of what sort it was)
scratching and scrambling about in the chimney close above her: then,
saying to herself “This is Bill,” she gave one sharp kick, and waited
to see what would happen next.

The first thing she heard was a general chorus of “There goes Bill!”
then the Rabbit’s voice along—“Catch him, you by the hedge!” then
silence, and then another confusion of voices—“Hold up his head—Brandy
now—Don’t choke him—How was it, old fellow? What happened to you? Tell
us all about it!”

### Review
The answer incorrectly describes Alice entering Wonderland rather than leaving it. Chunks 2 and 3 describe her following the Rabbit into the hole and falling down it, while none of the retrieved passages includes her waking up at the end. This example shows both a retrieval failure to find the relevant event and a generation error that confuses entering with leaving.

### Question: Who is Sherlock Holmes?

Answer: Sherlock Holmes is a fictional detective created by Sir Arthur Conan Doyle.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 407
Evidence: “Excuse me, I know everything of it. You are endeavouring to trace some
geese which were sold by Mrs. Oakshott, of Brixton Road, to a salesman
named Breckinridge, by him in turn to Mr. Windigate, of the Alpha, and
by him to his club, of which Mr. Henry Baker is a member.”

“Oh, sir, you are the very man whom I have longed to meet,” cried the
little fellow with outstretched hands and quivering fingers. “I can
hardly explain to you how interested I am in this matter.”

Sherlock Holmes hailed a four-wheeler which was passing. “In that case
we had better discuss it in a cosy room rather than in this wind-swept
market-place,” said he. “But pray tell me, before we go farther, who it
is that I have the pleasure of assisting.”

The man hesitated for an instant. “My name is John Robinson,” he
answered with a sidelong glance.

“No, no; the real name,” said Holmes sweetly. “It is always awkward
doing business with an alias.”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 425
Evidence: I had no keener pleasure than in following Holmes in his professional
investigations, and in admiring the rapid deductions, as swift as
intuitions, and yet always founded on a logical basis with which he
unravelled the problems which were submitted to him. I rapidly threw on
my clothes and was ready in a few minutes to accompany my friend down
to the sitting-room. A lady dressed in black and heavily veiled, who
had been sitting in the window, rose as we entered.

“Good-morning, madam,” said Holmes cheerily. “My name is Sherlock
Holmes. This is my intimate friend and associate, Dr. Watson, before
whom you can speak as freely as before myself. Ha! I am glad to see
that Mrs. Hudson has had the good sense to light the fire. Pray draw up
to it, and I shall order you a cup of hot coffee, for I observe that
you are shivering.”

“It is not cold which makes me shiver,” said the woman in a low voice,
changing her seat as requested.

“What, then?”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 155
Evidence: “Well, she had a slate-coloured, broad-brimmed straw hat, with a
feather of a brickish red. Her jacket was black, with black beads sewn
upon it, and a fringe of little black jet ornaments. Her dress was
brown, rather darker than coffee colour, with a little purple plush at
the neck and sleeves. Her gloves were greyish and were worn through at
the right forefinger. Her boots I didn’t observe. She had small round,
hanging gold earrings, and a general air of being fairly well-to-do in
a vulgar, comfortable, easy-going way.”

Sherlock Holmes clapped his hands softly together and chuckled.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 348
Evidence: A large and comfortable double-bedded room had been placed at our
disposal, and I was quickly between the sheets, for I was weary after
my night of adventure. Sherlock Holmes was a man, however, who, when he
had an unsolved problem upon his mind, would go for days, and even for
a week, without rest, turning it over, rearranging his facts, looking
at it from every point of view until he had either fathomed it or
convinced himself that his data were insufficient. It was soon evident
to me that he was now preparing for an all-night sitting. He took off
his coat and waistcoat, put on a large blue dressing-gown, and then
wandered about the room collecting pillows from his bed and cushions
from the sofa and armchairs. With these he constructed a sort of
Eastern divan, upon which he perched himself cross-legged, with an
ounce of shag tobacco and a box of matches laid out in front of him. In
the dim light of the lamp I saw him sitting there, an old briar pipe

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 229
Evidence: “Is a tall man, left-handed, limps with the right leg, wears
thick-soled shooting-boots and a grey cloak, smokes Indian cigars, uses
a cigar-holder, and carries a blunt pen-knife in his pocket. There are
several other indications, but these may be enough to aid us in our
search.”

Lestrade laughed. “I am afraid that I am still a sceptic,” he said.
“Theories are all very well, but we have to deal with a hard-headed
British jury.”

“_Nous verrons_,” answered Holmes calmly. “You work your own method,
and I shall work mine. I shall be busy this afternoon, and shall
probably return to London by the evening train.”

“And leave your case unfinished?”

“No, finished.”

“But the mystery?”

“It is solved.”

“Who was the criminal, then?”

“The gentleman I describe.”

“But who is he?”

“Surely it would not be difficult to find out. This is not such a
populous neighbourhood.”

### Review
The answer is factually correct, but it is not fully supported by the retrieved passages. Chunk 425 describes Holmes’s professional investigations and logical deductions, while chunks 229 and 348 further illustrate his problem-solving abilities. These passages support describing him as an investigator or detective, but they do not establish that he is a fictional character created by Sir Arthur Conan Doyle. The generated answer therefore includes information beyond what the retrieved passage text supports.

### Question: Who is Sherlock Holmes' friend and companion?

Answer: Sherlock Holmes' friend and companion is Dr. Watson.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 63
Evidence: I had called upon my friend, Mr. Sherlock Holmes, one day in the
 autumn of last year and found him in deep conversation with a very
 stout, florid-faced, elderly gentleman with fiery red hair. With an
 apology for my intrusion, I was about to withdraw when Holmes pulled
 me abruptly into the room and closed the door behind me.

“You could not possibly have come at a better time, my dear Watson,” he
said cordially.

“I was afraid that you were engaged.”

“So I am. Very much so.”

“Then I can wait in the next room.”

“Not at all. This gentleman, Mr. Wilson, has been my partner and helper
in many of my most successful cases, and I have no doubt that he will
be of the utmost use to me in yours also.”

The stout gentleman half rose from his chair and gave a bob of
greeting, with a quick little questioning glance from his small
fat-encircled eyes.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 425
Evidence: I had no keener pleasure than in following Holmes in his professional
investigations, and in admiring the rapid deductions, as swift as
intuitions, and yet always founded on a logical basis with which he
unravelled the problems which were submitted to him. I rapidly threw on
my clothes and was ready in a few minutes to accompany my friend down
to the sitting-room. A lady dressed in black and heavily veiled, who
had been sitting in the window, rose as we entered.

“Good-morning, madam,” said Holmes cheerily. “My name is Sherlock
Holmes. This is my intimate friend and associate, Dr. Watson, before
whom you can speak as freely as before myself. Ha! I am glad to see
that Mrs. Hudson has had the good sense to light the fire. Pray draw up
to it, and I shall order you a cup of hot coffee, for I observe that
you are shivering.”

“It is not cold which makes me shiver,” said the woman in a low voice,
changing her seat as requested.

“What, then?”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 368
Evidence: I had called upon my friend Sherlock Holmes upon the second morning
after Christmas, with the intention of wishing him the compliments of
the season. He was lounging upon the sofa in a purple dressing-gown, a
pipe-rack within his reach upon the right, and a pile of crumpled
morning papers, evidently newly studied, near at hand. Beside the couch
was a wooden chair, and on the angle of the back hung a very seedy and
disreputable hard-felt hat, much the worse for wear, and cracked in
several places. A lens and a forceps lying upon the seat of the chair
suggested that the hat had been suspended in this manner for the
purpose of examination.

“You are engaged,” said I; “perhaps I interrupt you.”

“Not at all. I am glad to have a friend with whom I can discuss my
results. The matter is a perfectly trivial one”—he jerked his thumb in
the direction of the old hat—“but there are points in connection with
it which are not entirely devoid of interest and even of instruction.”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 406
Evidence: “Ha! this may save us a visit to Brixton Road,” whispered Holmes. “Come
with me, and we will see what is to be made of this fellow.” Striding
through the scattered knots of people who lounged round the flaring
stalls, my companion speedily overtook the little man and touched him
upon the shoulder. He sprang round, and I could see in the gas-light
that every vestige of colour had been driven from his face.

“Who are you, then? What do you want?” he asked in a quavering voice.

“You will excuse me,” said Holmes blandly, “but I could not help
overhearing the questions which you put to the salesman just now. I
think that I could be of assistance to you.”

“You? Who are you? How could you know anything of the matter?”

“My name is Sherlock Holmes. It is my business to know what other
people don’t know.”

“But you can know nothing of this?”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 407
Evidence: “Excuse me, I know everything of it. You are endeavouring to trace some
geese which were sold by Mrs. Oakshott, of Brixton Road, to a salesman
named Breckinridge, by him in turn to Mr. Windigate, of the Alpha, and
by him to his club, of which Mr. Henry Baker is a member.”

“Oh, sir, you are the very man whom I have longed to meet,” cried the
little fellow with outstretched hands and quivering fingers. “I can
hardly explain to you how interested I am in this matter.”

Sherlock Holmes hailed a four-wheeler which was passing. “In that case
we had better discuss it in a cosy room rather than in this wind-swept
market-place,” said he. “But pray tell me, before we go farther, who it
is that I have the pleasure of assisting.”

The man hesitated for an instant. “My name is John Robinson,” he
answered with a sidelong glance.

“No, no; the real name,” said Holmes sweetly. “It is always awkward
doing business with an alias.”

### Review
The answer is correct and directly supported by chunk 425, where Holmes introduces Dr. Watson as his intimate friend and associate. Chunk 63 also supports their friendship and working relationship. Some other retrieved passages are less directly relevant, but sufficient evidence is available, and the generated answer is concise and grounded in it.

### Question: What do Alice and Caterpillar talk about?

Found 5 documents
Answer: Alice and the Caterpillar talk about Alice's confusion regarding her identity and changes throughout the day. They also discuss the concept of transformation, specifically how Alice has changed multiple times during the day.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 60
Evidence: “Well, perhaps your feelings may be different,” said Alice; “all I know
is, it would feel very queer to _me_.”

“You!” said the Caterpillar contemptuously. “Who are _you?_”

Which brought them back again to the beginning of the conversation.
Alice felt a little irritated at the Caterpillar’s making such _very_
short remarks, and she drew herself up and said, very gravely, “I
think, you ought to tell me who _you_ are, first.”

“Why?” said the Caterpillar.

Here was another puzzling question; and as Alice could not think of any
good reason, and as the Caterpillar seemed to be in a _very_ unpleasant
state of mind, she turned away.

“Come back!” the Caterpillar called after her. “I’ve something
important to say!”

This sounded promising, certainly: Alice turned and came back again.

“Keep your temper,” said the Caterpillar.

“Is that all?” said Alice, swallowing down her anger as well as she
could.

“No,” said the Caterpillar.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 59
Evidence: This was not an encouraging opening for a conversation. Alice replied,
rather shyly, “I—I hardly know, sir, just at present—at least I know
who I _was_ when I got up this morning, but I think I must have been
changed several times since then.”

“What do you mean by that?” said the Caterpillar sternly. “Explain
yourself!”

“I can’t explain _myself_, I’m afraid, sir,” said Alice, “because I’m
not myself, you see.”

“I don’t see,” said the Caterpillar.

“I’m afraid I can’t put it more clearly,” Alice replied very politely,
“for I can’t understand it myself to begin with; and being so many
different sizes in a day is very confusing.”

“It isn’t,” said the Caterpillar.

“Well, perhaps you haven’t found it so yet,” said Alice; “but when you
have to turn into a chrysalis—you will some day, you know—and then
after that into a butterfly, I should think you’ll feel it a little
queer, won’t you?”

“Not a bit,” said the Caterpillar.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 63
Evidence: “You are old,” said the youth, “one would hardly suppose
    That your eye was as steady as ever;
Yet you balanced an eel on the end of your nose—
    What made you so awfully clever?”

“I have answered three questions, and that is enough,”
    Said his father; “don’t give yourself airs!
Do you think I can listen all day to such stuff?
    Be off, or I’ll kick you down stairs!”

“That is not said right,” said the Caterpillar.

“Not _quite_ right, I’m afraid,” said Alice, timidly; “some of the
words have got altered.”

“It is wrong from beginning to end,” said the Caterpillar decidedly,
and there was silence for some minutes.

The Caterpillar was the first to speak.

“What size do you want to be?” it asked.

“Oh, I’m not particular as to size,” Alice hastily replied; “only one
doesn’t like changing so often, you know.”

“I _don’t_ know,” said the Caterpillar.

Alice said nothing: she had never been so much contradicted in her life
before, and she felt that she was losing her temper.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 65
Evidence: “One side of _what?_ The other side of _what?_” thought Alice to
herself.

“Of the mushroom,” said the Caterpillar, just as if she had asked it
aloud; and in another moment it was out of sight.

Alice remained looking thoughtfully at the mushroom for a minute,
trying to make out which were the two sides of it; and as it was
perfectly round, she found this a very difficult question. However, at
last she stretched her arms round it as far as they would go, and broke
off a bit of the edge with each hand.

“And now which is which?” she said to herself, and nibbled a little of
the right-hand bit to try the effect: the next moment she felt a
violent blow underneath her chin: it had struck her foot!

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 141
Evidence: “No, no! The adventures first,” said the Gryphon in an impatient tone:
“explanations take such a dreadful time.”

So Alice began telling them her adventures from the time when she first
saw the White Rabbit. She was a little nervous about it just at first,
the two creatures got so close to her, one on each side, and opened
their eyes and mouths so _very_ wide, but she gained courage as she
went on. Her listeners were perfectly quiet till she got to the part
about her repeating “_You are old, Father William_,” to the
Caterpillar, and the words all coming different, and then the Mock
Turtle drew a long breath, and said “That’s very curious.”

“It’s all about as curious as it can be,” said the Gryphon.

“It all came different!” the Mock Turtle repeated thoughtfully. “I
should like to hear her try and repeat something now. Tell her to
begin.” He looked at the Gryphon as if he thought it had some kind of
authority over Alice.

### Review
The answer accurately summarizes the main themes of their conversation. Chunk 59 supports Alice’s confusion about her identity and changing size, while chunk 63 describes her dislike of changing so often. The discussion of transformation is also supported by Alice’s comparison with the Caterpillar becoming a butterfly. However, “changing sizes and forms” is slightly imprecise because the passage distinguishes Alice’s size changes from the Caterpillar’s future transformation. Overall, the answer is largely supported, although chunk 141 concerns a later conversation and is less directly relevant.

### Question: What is the weather in Jakarta?

Answer: The weather in Jakarta is not mentioned in the given documents.

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 23
Evidence: So strange an accident has happened to us that I cannot forbear
recording it, although it is very probable that you will see me before
these papers can come into your possession.

Last Monday (July 31st) we were nearly surrounded by ice, which closed
in the ship on all sides, scarcely leaving her the sea-room in which
she floated. Our situation was somewhat dangerous, especially as we
were compassed round by a very thick fog. We accordingly lay to,
hoping that some change would take place in the atmosphere and weather.

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 8
Evidence: This is the most favourable period for travelling in Russia. They fly
quickly over the snow in their sledges; the motion is pleasant, and, in
my opinion, far more agreeable than that of an English stagecoach. The
cold is not excessive, if you are wrapped in furs—a dress which I have
already adopted, for there is a great difference between walking the
deck and remaining seated motionless for hours, when no exercise
prevents the blood from actually freezing in your veins. I have no
ambition to lose my life on the post-road between St. Petersburgh and
Archangel.

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 295
Evidence: “The old man, leaning on his son, walked each day at noon, when it did
not rain, as I found it was called when the heavens poured forth its
waters. This frequently took place, but a high wind quickly dried the
earth, and the season became far more pleasant than it had been.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 251
Evidence: It was in the latter days of September, and the equinoctial gales had
set in with exceptional violence. All day the wind had screamed and the
rain had beaten against the windows, so that even here in the heart of
great, hand-made London we were forced to raise our minds for the
instant from the routine of life and to recognise the presence of those
great elemental forces which shriek at mankind through the bars of his
civilisation, like untamed beasts in a cage. As evening drew in, the
storm grew higher and louder, and the wind cried and sobbed like a
child in the chimney. Sherlock Holmes sat moodily at one side of the
fireplace cross-indexing his records of crime, while I at the other was
deep in one of Clark Russell’s fine sea-stories until the howl of the
gale from without seemed to blend with the text, and the splash of the
rain to lengthen out into the long swash of the sea waves. My wife was
on a visit to her mother’s, and for a few days I was a dweller once

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 1
Evidence: I am already far north of London, and as I walk in the streets of
Petersburgh, I feel a cold northern breeze play upon my cheeks, which
braces my nerves and fills me with delight. Do you understand this
feeling? This breeze, which has travelled from the regions towards
which I am advancing, gives me a foretaste of those icy climes.
Inspirited by this wind of promise, my daydreams become more fervent
and vivid. I try in vain to be persuaded that the pole is the seat of
frost and desolation; it ever presents itself to my imagination as the
region of beauty and delight. There, Margaret, the sun is for ever
visible, its broad disk just skirting the horizon and diffusing a
perpetual splendour. There—for with your leave, my sister, I will put
some trust in preceding navigators—there snow and frost are banished;
and, sailing over a calm sea, we may be wafted to a land surpassing in
wonders and in beauty every region hitherto discovered on the habitable

### Review
The model appropriately states that it cannot determine Jakarta’s weather from the provided documents. Several retrieved passages discuss weather in other locations, but none answers the question. The model avoids presenting those descriptions as information about Jakarta. This example demonstrates successful abstention when the retrieved context does not contain the requested information.

# Experiment with Prompt Rewrite and Chat History

## What did I experiment with?
Creating a query-rewriting prompt and adding chat history.
## Why did I do this experiment?
To help the LLM answer follow-up questions from the user.
## How did I conduct this experiment?
By creating a prompt that instructs the LLM to rewrite follow-up questions containing references such as “she,” “there,” and “it” into standalone questions.
## What did I find?
Some questions were successfully rewritten in earlier tests. However, when the user entered another standalone question, the model sometimes rewrote it by connecting it to the previous conversation unnecessarily. In the latest test, it also generated answers instead of rewritten questions, sometimes introducing incorrect information. Therefore, the approach worked in some cases but did not consistently preserve the meaning of the user’s question.

## Result

Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

Question: Who does Alice follow down the rabbit hole?

Found 5 documents
Answer: Alice follows the White Rabbit down the rabbit hole.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 1
Evidence: CHAPTER I.     Down the Rabbit-Hole  CHAPTER II.    The Pool of Tears  CHAPTER III.   A Caucus-Race and a Long Tale  CHAPTER IV.    The Rabbit Sends in a Little Bill  CHAPTER V.     Advice from a Caterpillar  CHAPTER VI.    Pig and Pepper  CHAPTER VI...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 45
Evidence: “How queer it seems,” Alice said to herself, “to be going messages for a rabbit! I suppose Dinah’ll be sending me on messages next!” And she began fancying the sort of thing that would happen: “‘Miss Alice! Come here directly, and get ready for your ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 161
Evidence: Alice watched the White Rabbit as he fumbled over the list, feeling very curious to see what the next witness would be like, “—for they haven’t got much evidence _yet_,” she said to herself. Imagine her surprise, when the White Rabbit read out, at th...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Type Quit or quit to back to main menu.

Searching for: She follows the White Rabbit because he summons her into his lair in Wonderland.
Question: Why does she follow him?

Found 5 documents
Answer: She follows the White Rabbit because he summons her into his lair in Wonderland.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 44
Evidence: Very soon the Rabbit noticed Alice, as she went hunting about, and called out to her in an angry tone, “Why, Mary Ann, what _are_ you doing out here? Run home this moment, and fetch me a pair of gloves and a fan! Quick, now!” And Alice was so much fr...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 20
Evidence: After a time she heard a little pattering of feet in the distance, and she hastily dried her eyes to see what was coming. It was the White Rabbit returning, splendidly dressed, with a pair of white kid gloves in one hand and a large fan in the other:...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 45
Evidence: “How queer it seems,” Alice said to herself, “to be going messages for a rabbit! I suppose Dinah’ll be sending me on messages next!” And she began fancying the sort of thing that would happen: “‘Miss Alice! Come here directly, and get ready for your ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 43
Evidence: It was the White Rabbit, trotting slowly back again, and looking anxiously about as it went, as if it had lost something; and she heard it muttering to itself “The Duchess! The Duchess! Oh my dear paws! Oh my fur and whiskers! She’ll get me executed,...

Type Quit or quit to back to main menu.

Searching for: Yes, Alice eventually gets out of Wonderland when she reaches the end of the rabbit hole and emerges back into reality.
Question: Does she eventually get out of Wonderland?

Found 5 documents
Answer: Yes, Alice eventually gets out of Wonderland when she reaches the end of the rabbit hole and emerges back into reality.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 175
Evidence: Lastly, she pictured to herself how this same little sister of hers would, in the after-time, be herself a grown woman; and how she would keep, through all her riper years, the simple and loving heart of her childhood: and how she would gather about ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 89
Evidence: “I should like it very much,” said Alice, “but I haven’t been invited yet.”  “You’ll see me there,” said the Cat, and vanished.  Alice was not much surprised at this, she was getting so used to queer things happening. While she was looking at the pla...

Type Quit or quit to back to main menu.

Searching for: Alice gets out of Wonderland by reaching the end of the rabbit hole and emerging back into reality.
Question: How does she get out?

Found 5 documents
Answer: Alice gets out of Wonderland by reaching the end of the rabbit hole and emerging back into reality.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 8
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: aw...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 4
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.  Either the well was very deep...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself, “Oh dear! Oh dear! I shall be late!” (when she thought it over afterwards, it occurred to her that she ought to hav...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 175
Evidence: Lastly, she pictured to herself how this same little sister of hers would, in the after-time, be herself a grown woman; and how she would keep, through all her riper years, the simple and loving heart of her childhood: and how she would gather about ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 73
Evidence: “Well, be off, then!” said the Pigeon in a sulky tone, as it settled down again into its nest. Alice crouched down among the trees as well as she could, for her neck kept getting entangled among the branches, and every now and then she had to stop an...

Type Quit or quit to back to main menu.

Searching for: Sherlock Holmes is a fictional detective created by British author Sir Arthur Conan Doyle. He is known for solving complex cases using logical reasoning and deductive skills.
Question: Who is Sherlock Holmes?

Found 5 documents
Answer: Sherlock Holmes is a fictional detective created by British author Sir Arthur Conan Doyle. He is known for solving complex cases using logical reasoning and deductive skills.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 249
Evidence: When I glance over my notes and records of the Sherlock Holmes cases between the years ’82 and ’90, I am faced by so many which present strange and interesting features that it is no easy matter to know which to choose and which to leave. Some, howev...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 251
Evidence: The year ’87 furnished us with a long series of cases of greater or less interest, of which I retain the records. Among my headings under this one twelve months I find an account of the adventure of the Paradol Chamber, of the Amateur Mendicant Socie...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 422
Evidence: On glancing over my notes of the seventy odd cases in which I have during the last eight years studied the methods of my friend Sherlock Holmes, I find many tragic, some comic, a large number merely strange, but none commonplace; for, working as he d...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 282
Evidence: Sherlock Holmes closed his eyes and placed his elbows upon the arms of his chair, with his finger-tips together. “The ideal reasoner,” he remarked, “would, when he had once been shown a single fact in all its bearings, deduce from it not only all the...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 349
Evidence: A large and comfortable double-bedded room had been placed at our disposal, and I was quickly between the sheets, for I was weary after my night of adventure. Sherlock Holmes was a man, however, who, when he had an unsolved problem upon his mind, wou...

Type Quit or quit to back to main menu.

Searching for: His friend and companion is Dr. John Watson.
Question: Who is his friend and companion?

Found 5 documents
Answer: Dr. John Watson is Holmes' friend and companion mentioned in the passage.

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 143
Evidence: “You have made your position very clear to me,” said Holmes. “This is my friend, Dr. Watson, before whom you can speak as freely as before myself. Kindly tell us now all about your connection with Mr. Hosmer Angel.”

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 64
Evidence: I had called upon my friend, Mr. Sherlock Holmes, one day in the  autumn of last year and found him in deep conversation with a very  stout, florid-faced, elderly gentleman with fiery red hair. With an  apology for my intrusion, I was about to withdr...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 66
Evidence: “You did, Doctor, but none the less you must come round to my view, for otherwise I shall keep on piling fact upon fact on you until your reason breaks down under them and acknowledges me to be right. Now, Mr. Jabez Wilson here has been good enough t...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 19
Evidence: “You had my note?” he asked with a deep harsh voice and a strongly marked German accent. “I told you that I would call.” He looked from one to the other of us, as if uncertain which to address.  “Pray take a seat,” said Holmes. “This is my friend and...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 457
Evidence: “Fancy his having the insolence to confound me with the official detective force! This incident gives zest to our investigation, however, and I only trust that our little friend will not suffer from her imprudence in allowing this brute to trace her....

Type Quit or quit to back to main menu.

Searching for: Alice and the Caterpillar talk about tea and flowers.
Question: What do Alice and the Caterpillar talk about?

Found 5 documents
Answer: Alice and the Caterpillar talk about tea and flowers.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 61
Evidence: This was not an encouraging opening for a conversation. Alice replied, rather shyly, “I—I hardly know, sir, just at present—at least I know who I _was_ when I got up this morning, but I think I must have been changed several times since then.”  “What...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 62
Evidence: “Well, perhaps your feelings may be different,” said Alice; “all I know is, it would feel very queer to _me_.”  “You!” said the Caterpillar contemptuously. “Who are _you?_”  Which brought them back again to the beginning of the conversation. Alice fe...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 65
Evidence: “You are old,” said the youth, “one would hardly suppose     That your eye was as steady as ever; Yet you balanced an eel on the end of your nose—     What made you so awfully clever?”  “I have answered three questions, and that is enough,”     Said ...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 66
Evidence: “Are you content now?” said the Caterpillar.  “Well, I should like to be a _little_ larger, sir, if you wouldn’t mind,” said Alice: “three inches is such a wretched height to be.”  “It is a very good height indeed!” said the Caterpillar angrily, rear...

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 60
Evidence: The great question certainly was, what? Alice looked all round her at the flowers and the blades of grass, but she did not see anything that looked like the right thing to eat or drink under the circumstances. There was a large mushroom growing near ...

Type Quit or quit to back to main menu.

Searching for: The weather in Jakarta can vary depending on the time of year, but generally, it's hot and humid with temperatures ranging from 25°C to 30°C (77°F to 86°F) during the day and around 19°C to 24°C (66°F to 75°F) at night. It also experiences heavy rainfall throughout the year, especially during the monsoon season.
Question: What is the weather in Jakarta?

Found 5 documents
Answer: I do not know the answer to the question "The weather in Jakarta can vary depending on the time of year, but generally, it's hot and humid with temperatures ranging from 25°C to 30°C (77°F to 86°F) during the day and around 19°C to 24°C (66°F to 75°F) at night. It also experiences heavy rainfall throughout the year, especially during the monsoon season."

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 296
Evidence: “The old man, leaning on his son, walked each day at noon, when it did not rain, as I found it was called when the heavens poured forth its waters. This frequently took place, but a high wind quickly dried the earth, and the season became far more pl...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 24
Evidence: So strange an accident has happened to us that I cannot forbear recording it, although it is very probable that you will see me before these papers can come into your possession.  Last Monday (July 31st) we were nearly surrounded by ice, which closed...

Source: The Adventures of Sherlock Holmes by Arthur Conan Doyle –– Chunk 252
Evidence: It was in the latter days of September, and the equinoctial gales had set in with exceptional violence. All day the wind had screamed and the rain had beaten against the windows, so that even here in the heart of great, hand-made London we were force...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 267
Evidence: “One day, when I was oppressed by cold, I found a fire which had been left by some wandering beggars, and was overcome with delight at the warmth I experienced from it. In my joy I thrust my hand into the live embers, but quickly drew it out again wi...

Source: Frankenstein; or, the modern prometheus by Mary Wollstonecraft Shelley –– Chunk 537
Evidence: a great and sudden change. The sun might shine or the clouds might lower, but nothing could appear to me as it had done the day before. A fiend had snatched from me every hope of future happiness; no creature had ever been so miserable as I was; so f...

Type Quit or quit to back to main menu.

Goodbye

Menu:
1. Load Data
2. Ask LLM
3. Exit

Thank you for using Library AI Chatbot!

# Experiment with Similarity Search Threshold

## What did I experiment with?
Adding a similarity score threshold to the retrieval method.

## Why did I do this experiment?
In the baseline, the retriever still returned chunks for questions unrelated to the documents. Some answers were also incorrect because the retrieved chunks did not contain the information needed to answer the questions.

## How did I conduct this experiment?
I added the `score_threshold` parameter to the retrieval step. I tested thresholds of 0.7, 0.5, and 0.3.

## What did I find?
A threshold of 0.3 still returned documents for unrelated questions. At 0.7, no documents were retrieved for any of the example questions I tested. A threshold of 0.5 filtered out the retrieved documents for unrelated questions in my tests. However, for relevant questions, the retrieved documents remained similar to the results from Top-K retrieval without a threshold. These results suggest that a similarity threshold can help filter out unrelated results, but it does not guarantee better retrieval. Some retrieved documents still did not contain the information needed to answer the questions correctly.

## Result

Library AI Chatbot!

Menu:
1. Load Data
2. Ask LLM
3. Exit

Question: What is the weather in Jakarta?
No relevant docs were retrieved using the relevance score threshold 0.5

Found 0 documents
I couldn't find relevant information.

Type Quit or quit to back to main menu.
Question: How does Alice get out of Wonderland?

Found 5 documents
Answer: Alice gets out of Wonderland by climbing back through the rabbit hole.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 7
Evidence: Alice was not a bit hurt, and she jumped up on to her feet in a moment:
she looked up, but it was all dark overhead; before her was another
long passage, and the White Rabbit was still in sight, hurrying down
it. There was not a moment to be lost: away went Alice like the wind,
and was just in time to hear it say, as it turned a corner, “Oh my ears
and whiskers, how late it’s getting!” She was close behind it when she
turned the corner, but the Rabbit was no longer to be seen: she found
herself in a long, low hall, which was lit up by a row of lamps hanging
from the roof.

There were doors all round the hall, but they were all locked; and when
Alice had been all the way down one side and up the other, trying every
door, she walked sadly down the middle, wondering how she was ever to
get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 0
Evidence: Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence

CHAPTER I.
Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 2
Evidence: There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit actually _took a
watch out of its waistcoat-pocket_, and looked at it, and then hurried
on, Alice started to her feet, for it flashed across her mind that she
had never before seen a rabbit with either a waistcoat-pocket, or a
watch to take out of it, and burning with curiosity, she ran across the
field after it, and fortunately was just in time to see it pop down a
large rabbit-hole under the hedge.

In another moment down went Alice after it, never once considering how
in the world she was to get out again.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 3
Evidence: The rabbit-hole went straight on like a tunnel for some way, and then
dipped suddenly down, so suddenly that Alice had not a moment to think
about stopping herself before she found herself falling down a very
deep well.

Either the well was very deep, or she fell very slowly, for she had
plenty of time as she went down to look about her and to wonder what
was going to happen next. First, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-shelves; here and there she saw maps and pictures
hung upon pegs. She took down a jar from one of the shelves as she
passed; it was labelled “ORANGE MARMALADE”, but to her great
disappointment it was empty: she did not like to drop the jar for fear
of killing somebody underneath, so managed to put it into one of the
cupboards as she fell past it.

Source: Alice's Adventures in Wonderland by Lewis Carroll –– Chunk 51
Evidence: “Oh! So Bill’s got to come down the chimney, has he?” said Alice to
herself. “Shy, they seem to put everything upon Bill! I wouldn’t be in
Bill’s place for a good deal: this fireplace is narrow, to be sure; but
I _think_ I can kick a little!”

She drew her foot as far down the chimney as she could, and waited till
she heard a little animal (she couldn’t guess of what sort it was)
scratching and scrambling about in the chimney close above her: then,
saying to herself “This is Bill,” she gave one sharp kick, and waited
to see what would happen next.

The first thing she heard was a general chorus of “There goes Bill!”
then the Rabbit’s voice along—“Catch him, you by the hedge!” then
silence, and then another confusion of voices—“Hold up his head—Brandy
now—Don’t choke him—How was it, old fellow? What happened to you? Tell
us all about it!”

Type Quit or quit to back to main menu.

Goodbye

Menu:
1. Load Data
2. Ask LLM
3. Exit

Thank you for using Library AI Chatbot!

## Retrieval Evaluation

### Threshold = 0.7

Question: Why does Alice follow the White Rabbit?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 41 
3. alice_in_wonderland.txt — Chunk 2 (Referenced)

Hit: 1
Reciprocal rank: 0.333

Question: What opens the small door behind the curtain?

Hit: 0

Reciprocal rank: 0.000

Question: Why can't Alice climb the table leg to reach the key?

Hit: 0

Reciprocal rank: 0.000

Question: Who wakes Alice at the end of her adventure?

Hit: 0

Reciprocal rank: 0.000

Question: Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?

Hit: 0

Reciprocal rank: 0.000


Retrieval evaluation

Hit Rate: 20.00%

MRR: 0.067

### Threshold = 0.5

Question: Why does Alice follow the White Rabbit?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 41 
3. alice_in_wonderland.txt — Chunk 2 (Referenced)
4. alice_in_wonderland.txt — Chunk 158 
5. alice_in_wonderland.txt — Chunk 141 

Hit: 1

Reciprocal rank: 0.333

Question: What opens the small door behind the curtain?
1. alice_in_wonderland.txt — Chunk 8 (Referenced)
2. sherlock_holmes.txt — Chunk 723 
3. sherlock_holmes.txt — Chunk 52 
4. sherlock_holmes.txt — Chunk 645 
5. alice_in_wonderland.txt — Chunk 9 

Hit: 1

Reciprocal rank: 1.000

Question: Why can't Alice climb the table leg to reach the key?
1. alice_in_wonderland.txt — Chunk 13 (Referenced)
2. alice_in_wonderland.txt — Chunk 8 
3. alice_in_wonderland.txt — Chunk 48 
4. alice_in_wonderland.txt — Chunk 4 
5. alice_in_wonderland.txt — Chunk 3 

Hit: 1

Reciprocal rank: 1.000

Question: Who wakes Alice at the end of her adventure?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 51 
3. alice_in_wonderland.txt — Chunk 40 
4. alice_in_wonderland.txt — Chunk 96 
5. alice_in_wonderland.txt — Chunk 52 

Hit: 0

Reciprocal rank: 0.000

Question: Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?
1. sherlock_holmes.txt — Chunk 425 (Referenced)
2. sherlock_holmes.txt — Chunk 451 
3. sherlock_holmes.txt — Chunk 587 
4. sherlock_holmes.txt — Chunk 406 
5. sherlock_holmes.txt — Chunk 368 

Hit: 1

Reciprocal rank: 1.000

Retrieval evaluation

Hit Rate: 80.00%

MRR: 0.667

### Threshold 0.3

Question: Why does Alice follow the White Rabbit?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 41 
3. alice_in_wonderland.txt — Chunk 2 (Referenced)
4. alice_in_wonderland.txt — Chunk 158 
5. alice_in_wonderland.txt — Chunk 141 

Hit: 1

Reciprocal rank: 0.333

Question: What opens the small door behind the curtain?
1. alice_in_wonderland.txt — Chunk 8 (Referenced)
2. sherlock_holmes.txt — Chunk 723 
3. sherlock_holmes.txt — Chunk 52 
4. sherlock_holmes.txt — Chunk 645 
5. alice_in_wonderland.txt — Chunk 9 

Hit: 1

Reciprocal rank: 1.000

Question: Why can't Alice climb the table leg to reach the key?
1. alice_in_wonderland.txt — Chunk 13 (Referenced)
2. alice_in_wonderland.txt — Chunk 8 
3. alice_in_wonderland.txt — Chunk 48 
4. alice_in_wonderland.txt — Chunk 4 
5. alice_in_wonderland.txt — Chunk 3 

Hit: 1

Reciprocal rank: 1.000

Question: Who wakes Alice at the end of her adventure?
1. alice_in_wonderland.txt — Chunk 7 
2. alice_in_wonderland.txt — Chunk 51 
3. alice_in_wonderland.txt — Chunk 40 
4. alice_in_wonderland.txt — Chunk 96 
5. alice_in_wonderland.txt — Chunk 52 

Hit: 0

Reciprocal rank: 0.000

Question: Who does Sherlock Holmes introduce as his friend and associate to the veiled lady?
1. sherlock_holmes.txt — Chunk 425 (Referenced)
2. sherlock_holmes.txt — Chunk 451 
3. sherlock_holmes.txt — Chunk 587 
4. sherlock_holmes.txt — Chunk 406 
5. sherlock_holmes.txt — Chunk 368 

Hit: 1

Reciprocal rank: 1.000


Retrieval evaluation

Hit Rate: 80.00%

MRR: 0.667

## Generation Evaluation

### Threshold = 0.7

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>user_input</th>
      <th>response</th>
      <th>reference</th>
      <th>faithfulness</th>
      <th>answer_correctness</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Why does Alice follow the White Rabbit?</td>
      <td>Alice follows the White Rabbit because he is r...</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>1.0</td>
      <td>0.559342</td>
    </tr>
    <tr>
      <th>1</th>
      <td>What opens the small door behind the curtain?</td>
      <td>I couldn't find relevant information.</td>
      <td>A little golden key.</td>
      <td>0.0</td>
      <td>0.106033</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Why can't Alice climb the table leg to reach t...</td>
      <td>I couldn't find relevant information.</td>
      <td>Because the table leg is too slippery.</td>
      <td>0.0</td>
      <td>0.106269</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Who wakes Alice at the end of her adventure?</td>
      <td>I couldn't find relevant information.</td>
      <td>Her sister.</td>
      <td>0.0</td>
      <td>0.087461</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Who does Sherlock Holmes introduce as his frie...</td>
      <td>I couldn't find relevant information.</td>
      <td>Dr. Watson.</td>
      <td>0.0</td>
      <td>0.104071</td>
    </tr>
  </tbody>
</table>
</div>

Average scores:

faithfulness          0.200000

answer_correctness    0.192635

### Threshold = 0.5

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>user_input</th>
      <th>response</th>
      <th>reference</th>
      <th>faithfulness</th>
      <th>answer_correctness</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Why does Alice follow the White Rabbit?</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>1.000000</td>
      <td>0.717572</td>
    </tr>
    <tr>
      <th>1</th>
      <td>What opens the small door behind the curtain?</td>
      <td>Alice opens the small door behind the curtain.</td>
      <td>A little golden key.</td>
      <td>1.000000</td>
      <td>0.899673</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Why can't Alice climb the table leg to reach t...</td>
      <td>Alice cannot climb the table leg to reach the ...</td>
      <td>Because the table leg is too slippery.</td>
      <td>0.000000</td>
      <td>0.656221</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Who wakes Alice at the end of her adventure?</td>
      <td>Alice doesn't wake anyone at the end of her ad...</td>
      <td>Her sister.</td>
      <td>0.333333</td>
      <td>0.090247</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Who does Sherlock Holmes introduce as his frie...</td>
      <td>Sherlock Holmes introduces Dr. Watson as his f...</td>
      <td>Dr. Watson.</td>
      <td>1.000000</td>
      <td>0.914123</td>
    </tr>
  </tbody>
</table>
</div>

Average scores:

faithfulness          0.666667

answer_correctness    0.655567

### Threshold = 0.3

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>user_input</th>
      <th>response</th>
      <th>reference</th>
      <th>faithfulness</th>
      <th>answer_correctness</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Why does Alice follow the White Rabbit?</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>Alice follows the White Rabbit because she is ...</td>
      <td>1.000000</td>
      <td>0.717572</td>
    </tr>
    <tr>
      <th>1</th>
      <td>What opens the small door behind the curtain?</td>
      <td>Alice opens the small door behind the curtain.</td>
      <td>A little golden key.</td>
      <td>1.000000</td>
      <td>0.899673</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Why can't Alice climb the table leg to reach t...</td>
      <td>Alice cannot climb the table leg to reach the ...</td>
      <td>Because the table leg is too slippery.</td>
      <td>0.000000</td>
      <td>0.656221</td>
    </tr>
    <tr>
      <th>3</th>
      <td>Who wakes Alice at the end of her adventure?</td>
      <td>Alice doesn't wake anyone at the end of her ad...</td>
      <td>Her sister.</td>
      <td>0.333333</td>
      <td>0.090247</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Who does Sherlock Holmes introduce as his frie...</td>
      <td>Sherlock Holmes introduces Dr. Watson as his f...</td>
      <td>Dr. Watson.</td>
      <td>1.000000</td>
      <td>0.914123</td>
    </tr>
  </tbody>
</table>
</div>

Average scores:

faithfulness          0.666667

answer_correctness    0.655567